In [1]:
# @title 🚗 Model Test Demo: Kök Neden Analizi
# Bu not defteri, daha önce eğitilen BERT modelini yükler ve örnek verilerle test eder.

# Gerekli kütüphaneleri yükleyelim
!pip install transformers torch pandas scikit-learn --quiet

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import os
from google.colab import drive

# ---------------------------------------------------------
# 1. DRIVE BAĞLANTISI (Model Drive'da olduğu için)
# ---------------------------------------------------------
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# ---------------------------------------------------------
# 2. AYARLAR
# ---------------------------------------------------------
# NOT: GitHub'dan indirenler için buradaki yolları değiştirmeleri gerekebilir.
# Biz şu an Drive üzerindeki konumunu kullanıyoruz.
model_path = '/content/drive/MyDrive/Bottleneck/Kayitli_Modeller/BERT_Final_Classifier'
data_path = '/content/drive/MyDrive/Bottleneck/RECALL_VERISI_5000_TR.csv'

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Çalışma Ortamı: {device.upper()}")

# ---------------------------------------------------------
# 3. KAYITLI MODELİ YÜKLEME
# ---------------------------------------------------------
if os.path.exists(model_path):
    print(f"📂 Model yükleniyor...")
    try:
        # Uzun metinlerde hata almamak için truncation aktif ediyoruz
        tokenizer = AutoTokenizer.from_pretrained(model_path, model_max_length=512)
        model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)

        # Pipeline oluştur
        classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0 if device=="cuda" else -1)
        print("✅ Model başarıyla yüklendi ve analize hazır!")
    except Exception as e:
        print(f"❌ Model yüklenirken hata: {e}")
else:
    print("❌ Model dosyası bulunamadı. Lütfen yolu kontrol edin.")

# ---------------------------------------------------------
# 4. CANLI TEST (DEMO)
# ---------------------------------------------------------
# Veri setinden rastgele 5 örnek çekip modele soralım
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    df = df.dropna(subset=['Kök_Neden_TR'])

    # Rastgele 5 örnek seç
    samples = df.sample(n=5, random_state=42)

    print("\n--- 🔍 MODEL TAHMİNLERİ ---")

    for idx, row in samples.iterrows():
        text = row['Kök_Neden_TR']
        # Tahmin (Truncation ile)
        result = classifier(text, truncation=True, max_length=512)[0]

        print(f"\n📝 ŞİKAYET: {text[:150]}...")
        print(f"🤖 TAHMİN: {result['label']}")
        print(f"📊 GÜVEN: %{result['score']*100:.2f}")
        print("-" * 50)
else:
    print("Veri seti bulunamadı, manuel test yapılıyor...")
    # Veri seti yoksa manuel örnek
    text = "Frene basınca araç titriyor ve ses geliyor."
    result = classifier(text)[0]
    print(f"\nÖrnek Metin: {text}")
    print(f"Tahmin: {result['label']}")

Mounted at /content/drive
✅ Çalışma Ortamı: CPU
📂 Model yükleniyor...


Device set to use cpu


✅ Model başarıyla yüklendi ve analize hazır!

--- 🔍 MODEL TAHMİNLERİ ---

📝 ŞİKAYET: 8,1 L V8 MOTOR İLE DONANIMLI BAZI PIPE KAMYONLARDA, KRANK MİLİ KONUMU ARALIKLI ÇALIŞABİLİR VEYA TAMAMEN ARIZALI OLABİLİR.  SENSÖR ARALIKLI ÇALIŞIRSA S...
🤖 TAHMİN: Motor ve Güç Aktarma
📊 GÜVEN: %99.31
--------------------------------------------------

📝 ŞİKAYET: BOJİNİN ÖN AKSINA MONTAJI YAPILAN TİŞÖRT BRAKETİ, YOL YÜZEYİ DÜZENSİZLİKLERİNDEN KAYNAKLANAN TİTREŞİM NEDENİYLE BOZULABİLİR. BRAKET ARIZASI TİŞÖRT BRA...
🤖 TAHMİN: Fren Sistemi
📊 GÜVEN: %98.26
--------------------------------------------------

📝 ŞİKAYET: Toyota Motor Engineering & Manufacturing (Toyota), Panoramik Görüntü Monitörü (PVM) sistemi ile donatılmış bazı 2022-2026 Toyota, Lexus ve Subaru Solt...
🤖 TAHMİN: Motor ve Güç Aktarma
📊 GÜVEN: %99.41
--------------------------------------------------

📝 ŞİKAYET: FOREST RIVER, 176 2009 kardinal beşinci tekerlek römorkunu geri çağırıyor.  BU RÖMORKLAR FEDERAL MOTORLU TAŞIT GÜVENLİK STANDARDI N